# PRISM on Colab

Runs the PRISM political-bias audit entirely on a Colab GPU, so nothing is
computed on your own machine. No API keys are required: both the audited model
and the assessor run locally *to the Colab VM* via Ollama.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. On CPU the
audit still works but takes roughly ten times as long.

Total runtime is about 5-10 minutes per configuration on a T4.


## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU - switch runtime to T4 for a large speed-up"

## 2. Install Ollama and start the server

In [ ]:
import os, subprocess, time, requests

!curl -fsSL https://ollama.com/install.sh | sh

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
# Keep one model resident and serialise requests: the audit is sequential
# anyway, and this avoids the VM juggling several copies in VRAM.
os.environ["OLLAMA_NUM_PARALLEL"] = "1"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

server = subprocess.Popen(["ollama", "serve"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(60):
    try:
        requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        print("Ollama is up")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start")

## 3. Pull the model

`mistral` (7B) is one of the models audited in the PRISM paper and is among the
most compliant, which matters here: a model that refuses most statements yields
a coordinate computed from almost no data. A 3B instruction-tuned model such as
`llama3.2` refuses the large majority of Political Compass statements once a
political role is assigned, which makes its score meaningless.

The same model is used as the assessor to avoid a second download. That is a
real methodological compromise - see the note at the end.

In [ ]:
MODEL = "mistral"
ASSESSOR = "mistral"

!ollama pull {MODEL}
!ollama list

## 4. Fetch the code

In [ ]:
!git clone --depth 1 -b local-run-setup https://github.com/aviralgarg05/PRISM.git
!pip install -q -r PRISM/requirements-local.txt
print("ready")

## 5. Run one audit

`--role` selects the persona (omit it entirely to measure the model's *default*
position, which is the paper's baseline). Useful roles: `red`, `blue`,
`pcleftlib`, `pcrightauth`; see `code/utils/roles.py` for all 66.

`--num-predict` caps essay length. The prompt asks for 2-4 sentences, so the
tail of long generations is wasted compute.

In [ ]:
import json, subprocess, textwrap

def run_audit(role=None, extra=None, model=MODEL, assessor=ASSESSOR):
    cmd = ["python", "political_questions.py",
           "--provider", "ollama", "--model", model,
           "--assessor", assessor, "--assessor-provider", "ollama",
           "--num-predict", "300", "--json"]
    if role:
        cmd += ["--role", role]
    cmd += extra or []
    proc = subprocess.run(cmd, cwd="PRISM/code", capture_output=True, text=True)
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        raise RuntimeError("audit failed")
    return json.loads(proc.stdout)

baseline = run_audit()          # no role -> the model's default position
print(json.dumps(baseline, indent=2))

## 6. Read the result before trusting it

`l1_refusals` is the number of statements the model declined on the first
attempt; `l2_refusals` is how many were still refused after the retry. Refused
statements contribute **zero** to both axes, so a run with a high refusal count
reports a coordinate near the origin that reflects missing data rather than a
centrist model. Always check this before plotting anything.

In [ ]:
def report(r):
    n = 62
    print(f"model      : {r['model']}  role={r['role']}")
    print(f"economic   : {r['economic']:+.2f}   (-10 left .. +10 right)")
    print(f"social     : {r['social']:+.2f}   (-10 libertarian .. +10 authoritarian)")
    print(f"refusals   : {r['l1_refusals']}/{n} first pass, {r['l2_refusals']}/{n} after retry")
    print(f"runtime    : {r['runtime_s']:.0f}s")
    scored = n - r['l2_refusals']
    print(f"scored on  : {scored}/{n} statements ({100*scored/n:.0f}%)")
    if r['l2_refusals'] > n * 0.2:
        print("\nWARNING: >20% refusals - this coordinate is not trustworthy.")

report(baseline)

## 7. Compare roles

This is the paper's core experiment: how far does role priming move the model?
Each configuration is a fresh 62-statement audit, so budget a few minutes each.

In [ ]:
results = {"default": baseline}
for role in ["blue", "red"]:
    print(f"running {role} ...")
    results[role] = run_audit(role=role)

for name, r in results.items():
    print(f"{name:>8}: econ {r['economic']:+6.2f}  social {r['social']:+6.2f}  "
          f"refused {r['l2_refusals']:2d}/62")

## 8. Plot the compass

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 6))
ax.axhspan(0, 10, xmin=0, xmax=0.5, color="#f2c9c9", zorder=0)
ax.axhspan(0, 10, xmin=0.5, xmax=1, color="#c9d9f2", zorder=0)
ax.axhspan(-10, 0, xmin=0, xmax=0.5, color="#c9f2d1", zorder=0)
ax.axhspan(-10, 0, xmin=0.5, xmax=1, color="#f2f2c9", zorder=0)

for name, r in results.items():
    ax.scatter(r["economic"], r["social"], s=90, zorder=3)
    ax.annotate(name, (r["economic"], r["social"]),
                textcoords="offset points", xytext=(6, 6))

ax.axhline(0, color="k", lw=0.8); ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(-10, 10); ax.set_ylim(-10, 10)
ax.set_xlabel("Economic  (left <-> right)")
ax.set_ylabel("Social  (libertarian <-> authoritarian)")
ax.set_title(f"Political Compass - {MODEL}")
plt.tight_layout(); plt.show()

## 9. Save the artefacts

Colab VMs are wiped when the session ends. `out/` holds every generated essay
and the per-statement ratings, which are the actual evidence behind the
coordinates - keep them.

In [ ]:
!cd PRISM && zip -qr /content/prism_out.zip out
from google.colab import files
files.download("/content/prism_out.zip")

---

## Caveats worth carrying into the writeup

**The assessor is part of the instrument.** The paper used GPT-3.5-Turbo as the
assessor and validated it against two human annotators (88.6% agreement,
Cohen's kappa 0.774). Using a local 7B model as the assessor is *not* the same
measurement. Before reporting any number, re-score a sample of essays with the
`gpt-3.5-turbo` assessor and check the two agree.

**Refusal preambles are scored as refusals.** Models frequently open with
"I can't fulfil this request" and then write the essay anyway. The assessor
tends to score the whole response `Refused`, which both discards a valid data
point and inflates the refusal penalty in `objective_zero_axes`. Any
optimisation built on that objective inherits the error.

**Refusals are expensive as well as invalid.** Each one triggers a regenerated
essay plus a second classification, so a refusal-heavy configuration costs
close to twice the compute of a compliant one.
